# Jaguar Re-Identification Challenge - Data Cleaning & Preprocessing

**Following EDA**: image cleaning and formatting for training  
**Issues found**: RGBA (4 channels), highly variable sizes, class imbalance  
**Target**: RGB 224x224 images, encoded labels, train/val split

In [ ]:
import os
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = os.path.join('..', 'data')
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train', 'train')
TEST_IMG_DIR = os.path.join(DATA_DIR, 'test', 'test')

CLEAN_DIR = os.path.join(DATA_DIR, 'cleaned')
CLEAN_TRAIN_DIR = os.path.join(CLEAN_DIR, 'train')
CLEAN_TEST_DIR = os.path.join(CLEAN_DIR, 'test')

TARGET_SIZE = (224, 224)

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train images: {len(train_df)}")
print(f"Test pairs: {len(test_df)}")
print(f"Target size: {TARGET_SIZE}")

## 1. Check for corrupted images

In [ ]:
corrupted_train = []
train_info = []

for filename in train_df['filename']:
    img_path = os.path.join(TRAIN_IMG_DIR, filename)
    try:
        img = Image.open(img_path)
        img.verify()
        img = Image.open(img_path)
        w, h = img.size
        train_info.append({'filename': filename, 'width': w, 'height': h, 'mode': img.mode, 'valid': True})
    except Exception as e:
        corrupted_train.append((filename, str(e)))
        train_info.append({'filename': filename, 'width': 0, 'height': 0, 'mode': 'N/A', 'valid': False})

corrupted_test = []
test_files = sorted(set(test_df['query_image'].tolist() + test_df['gallery_image'].tolist()))

for filename in test_files:
    img_path = os.path.join(TEST_IMG_DIR, filename)
    try:
        img = Image.open(img_path)
        img.verify()
    except Exception as e:
        corrupted_test.append((filename, str(e)))

print(f"=== Corruption check ===")
print(f"Train - corrupted: {len(corrupted_train)} / {len(train_df)}")
print(f"Test  - corrupted: {len(corrupted_test)} / {len(test_files)}")

if corrupted_train:
    print(f"\nCorrupted train files:")
    for f, e in corrupted_train:
        print(f"  {f}: {e}")
if corrupted_test:
    print(f"\nCorrupted test files:")
    for f, e in corrupted_test:
        print(f"  {f}: {e}")

## 2. Image modes (RGBA, RGB, etc.)

In [ ]:
info_df = pd.DataFrame(train_info)
mode_counts = info_df['mode'].value_counts()

print("=== Image modes (train) ===")
print(mode_counts)
print(f"\nRGBA -> RGB needed: {(info_df['mode'] == 'RGBA').sum()}")
print(f"Already RGB: {(info_df['mode'] == 'RGB').sum()}")
print(f"Grayscale (L): {(info_df['mode'] == 'L').sum()}")

## 3. Duplicate detection

In [ ]:
def file_hash(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

train_hashes = {}
duplicates = []

for filename in train_df['filename']:
    img_path = os.path.join(TRAIN_IMG_DIR, filename)
    h = file_hash(img_path)
    if h in train_hashes:
        duplicates.append((filename, train_hashes[h]))
    else:
        train_hashes[h] = filename

print(f"=== Duplicate check ===")
print(f"Exact duplicate pairs: {len(duplicates)}")
if duplicates:
    for dup, orig in duplicates[:10]:
        dup_label = train_df[train_df['filename'] == dup]['ground_truth'].values[0]
        orig_label = train_df[train_df['filename'] == orig]['ground_truth'].values[0]
        print(f"  {dup} ({dup_label}) == {orig} ({orig_label})")

## 4. Outlier images (very small or extreme aspect ratio)

In [ ]:
valid_info = info_df[info_df['valid']].copy()
valid_info['aspect'] = valid_info['width'] / valid_info['height']

small_images = valid_info[(valid_info['width'] < 100) | (valid_info['height'] < 100)]
print(f"=== Outlier images ===")
print(f"Very small (< 100px): {len(small_images)}")
if len(small_images) > 0:
    print(small_images[['filename', 'width', 'height']].to_string(index=False))

extreme_aspect = valid_info[(valid_info['aspect'] > 5) | (valid_info['aspect'] < 0.2)]
print(f"\nExtreme aspect ratio (>5 or <0.2): {len(extreme_aspect)}")
if len(extreme_aspect) > 0:
    for _, row in extreme_aspect.iterrows():
        label = train_df[train_df['filename'] == row['filename']]['ground_truth'].values[0]
        print(f"  {row['filename']}: {row['width']}x{row['height']} (ratio={row['aspect']:.2f}) - {label}")

In [ ]:
outliers = pd.concat([small_images, extreme_aspect]).drop_duplicates(subset='filename')

if len(outliers) > 0:
    n_show = min(8, len(outliers))
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle('Outlier images (small or extreme aspect ratio)', fontsize=14)
    axes = axes.flatten()

    for i in range(n_show):
        row = outliers.iloc[i]
        img = Image.open(os.path.join(TRAIN_IMG_DIR, row['filename']))
        label = train_df[train_df['filename'] == row['filename']]['ground_truth'].values[0]
        axes[i].imshow(img)
        axes[i].set_title(f"{row['filename']}\n{row['width']}x{row['height']} - {label}", fontsize=8)
        axes[i].axis('off')
    for i in range(n_show, 8):
        axes[i].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join('..', 'outputs', 'outlier_images.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No outlier images found.")

## 5. Convert and resize

In [ ]:
def clean_and_resize(img_path, target_size=TARGET_SIZE):
    img = Image.open(img_path)

    if img.mode == 'RGBA':
        bg = Image.new('RGB', img.size, (255, 255, 255))
        bg.paste(img, mask=img.split()[3])
        img = bg
    elif img.mode != 'RGB':
        img = img.convert('RGB')

    img = img.resize(target_size, Image.LANCZOS)
    return img

# Quick test
test_img = clean_and_resize(os.path.join(TRAIN_IMG_DIR, train_df['filename'].iloc[0]))
print(f"Test: size={test_img.size}, mode={test_img.mode}")

In [ ]:
os.makedirs(CLEAN_TRAIN_DIR, exist_ok=True)
os.makedirs(CLEAN_TEST_DIR, exist_ok=True)

print("Processing train images...")
train_errors = []
for i, filename in enumerate(train_df['filename']):
    src = os.path.join(TRAIN_IMG_DIR, filename)
    dst = os.path.join(CLEAN_TRAIN_DIR, filename)
    try:
        img = clean_and_resize(src)
        img.save(dst)
    except Exception as e:
        train_errors.append((filename, str(e)))
    if (i + 1) % 500 == 0:
        print(f"  {i + 1}/{len(train_df)} done")

print(f"Train: {len(train_df) - len(train_errors)}/{len(train_df)} processed")
if train_errors:
    for f, e in train_errors:
        print(f"  ERROR {f}: {e}")

In [ ]:
print("Processing test images...")
test_errors = []
for i, filename in enumerate(test_files):
    src = os.path.join(TEST_IMG_DIR, filename)
    dst = os.path.join(CLEAN_TEST_DIR, filename)
    try:
        img = clean_and_resize(src)
        img.save(dst)
    except Exception as e:
        test_errors.append((filename, str(e)))
    if (i + 1) % 100 == 0:
        print(f"  {i + 1}/{len(test_files)} done")

print(f"Test: {len(test_files) - len(test_errors)}/{len(test_files)} processed")
if test_errors:
    for f, e in test_errors:
        print(f"  ERROR {f}: {e}")

## 6. Verify cleaned images

In [ ]:
clean_sizes = set()
clean_modes = set()

for filename in train_df['filename']:
    img = Image.open(os.path.join(CLEAN_TRAIN_DIR, filename))
    clean_sizes.add(img.size)
    clean_modes.add(img.mode)

print("=== Cleaned train images ===")
print(f"Unique sizes: {clean_sizes}")
print(f"Unique modes: {clean_modes}")
print(f"All uniform? {len(clean_sizes) == 1 and len(clean_modes) == 1}")

In [ ]:
sample_filenames = train_df.sample(4, random_state=42)['filename'].tolist()

fig, axes = plt.subplots(4, 2, figsize=(10, 16))
fig.suptitle('Original vs Cleaned (224x224 RGB)', fontsize=14)

for i, filename in enumerate(sample_filenames):
    orig = Image.open(os.path.join(TRAIN_IMG_DIR, filename))
    cleaned = Image.open(os.path.join(CLEAN_TRAIN_DIR, filename))
    label = train_df[train_df['filename'] == filename]['ground_truth'].values[0]

    axes[i][0].imshow(orig)
    axes[i][0].set_title(f'Original: {orig.size} {orig.mode}\n{label}', fontsize=9)
    axes[i][0].axis('off')

    axes[i][1].imshow(cleaned)
    axes[i][1].set_title(f'Cleaned: {cleaned.size} {cleaned.mode}\n{label}', fontsize=9)
    axes[i][1].axis('off')

plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'original_vs_cleaned.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Label encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['ground_truth'])

label_mapping = pd.DataFrame({
    'identity': le.classes_,
    'label': range(len(le.classes_))
})

print(f"=== Label encoding ===")
print(f"Number of classes: {len(le.classes_)}")
print(f"\nMapping:")
print(label_mapping.to_string(index=False))

In [ ]:
clean_train_csv = train_df[['filename', 'ground_truth', 'label']].copy()
clean_train_csv.to_csv(os.path.join(CLEAN_DIR, 'train_clean.csv'), index=False)
label_mapping.to_csv(os.path.join(CLEAN_DIR, 'label_mapping.csv'), index=False)

print(f"Saved: {os.path.join(CLEAN_DIR, 'train_clean.csv')}")
print(f"Saved: {os.path.join(CLEAN_DIR, 'label_mapping.csv')}")
display(clean_train_csv.head(10))

## 8. Split train/validation

In [ ]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(
    clean_train_csv,
    test_size=0.2,
    random_state=42,
    stratify=clean_train_csv['label']
)

print(f"=== Train/Validation split ===")
print(f"Train: {len(train_split)} images")
print(f"Val:   {len(val_split)} images")
print(f"\nIdentities in train: {train_split['label'].nunique()}")
print(f"Identities in val:   {val_split['label'].nunique()}")

train_split.to_csv(os.path.join(CLEAN_DIR, 'train_split.csv'), index=False)
val_split.to_csv(os.path.join(CLEAN_DIR, 'val_split.csv'), index=False)

print(f"\nSaved: {os.path.join(CLEAN_DIR, 'train_split.csv')}")
print(f"Saved: {os.path.join(CLEAN_DIR, 'val_split.csv')}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

train_counts = train_split['ground_truth'].value_counts().sort_index()
val_counts = val_split['ground_truth'].value_counts().sort_index()

x = range(len(train_counts))
width = 0.35

axes[0].bar([i - width/2 for i in x], train_counts.values, width, label='Train', color='steelblue')
axes[0].bar([i + width/2 for i in x], val_counts.values, width, label='Val', color='coral')
axes[0].set_xlabel('Jaguar identity')
axes[0].set_ylabel('Number of images')
axes[0].set_title('Train vs Validation split by identity')
axes[0].legend()
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(train_counts.index, rotation=90, fontsize=7)

train_pct = train_counts / (train_counts + val_counts) * 100
axes[1].bar(range(len(train_pct)), train_pct.values, color='steelblue')
axes[1].axhline(y=80, color='red', linestyle='--', label='Target: 80%')
axes[1].set_xlabel('Jaguar identity (index)')
axes[1].set_ylabel('Train proportion (%)')
axes[1].set_title('Train proportion per identity (target: 80%)')
axes[1].set_ylim(0, 100)
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join('..', 'outputs', 'train_val_split.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Summary

In [ ]:
print("=" * 60)
print("DATA CLEANING - SUMMARY")
print("=" * 60)
print(f"""  
DATASET:
  - Corrupted images: {len(corrupted_train)} train, {len(corrupted_test)} test
  - Exact duplicates: {len(duplicates)}
  - Outliers (aspect ratio >5): {len(extreme_aspect)}

PREPROCESSING:
  - RGBA -> RGB (white background)
  - Resize {TARGET_SIZE[0]}x{TARGET_SIZE[1]} Lanczos
  - {len(le.classes_)} classes encoded (0-{len(le.classes_) - 1})

SPLIT:
  - Train: {len(train_split)} (80%) / Val: {len(val_split)} (20%)
  - Stratified by identity

OUTPUT (data/cleaned/):
  - train/, test/       : images {TARGET_SIZE[0]}x{TARGET_SIZE[1]} RGB
  - train_clean.csv     : filename + ground_truth + label
  - label_mapping.csv   : identity <-> label
  - train_split.csv     : 80% train
  - val_split.csv       : 20% val
""")